In [ ]:
pip install colorednoise

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.fftpack import fft, fftfreq
from scipy.signal import get_window
import colorednoise as cn

# Função de Envelope Flare
def flare_curve(t, t_peak, rise_time, decay_time):
    rise = np.exp(-(t_peak - t)**2 / (2 * rise_time**2))
    decay = np.exp(-(t - t_peak) / decay_time)
    return np.where(t < t_peak, rise, decay)

# Gerador de Sinal Otimizado (Separa a geração do loop de janelas)
def generate_signal_base(LEAK, N, fs, noise_amp):
    T = 1.0 / fs
    df = 1 / (N * T)
    chopper_freq = 20.0
    rede_freq = 60.0

    FN1 = np.round(chopper_freq * N * T)
    if LEAK:
        FN1 += 0.5

    chopper = FN1 * df
    x = np.linspace(0.0, N * T, N, endpoint=False)

    t_peak, rise_time, decay_time = 10, 5, 10
    flare = flare_curve(x, t_peak, rise_time, decay_time)

    y0 = np.sin(chopper * 2.0 * np.pi * x) * flare
    phase = np.random.rand(1)
    y1 = 0.2 * np.sin(rede_freq * 2.0 * np.pi * x - phase)

    noise_white = np.random.normal(0.0, 1.0, N)
    pink_noise = cn.powerlaw_psd_gaussian(1, N)
    delta_ampl = 0.5 * np.cos(2.0 * np.pi * 2.0 * x)

    y = (y0 + y1 + noise_white * noise_amp + pink_noise * noise_amp) + delta_ampl
    return x, y, flare

# Parâmetros Globais
windows = ['rectangular', 'hamming', 'hann', 'bartlett', 'blackman', 'flattop']
window_names = ['Rectangular', 'Hamming', 'Hann', 'Bartlett', 'Blackman', 'Flat-top']
correction_calc = [1.0, 1.85, 2.0, 2.02, 2.40, 4.68]
noise_levels = [0.1, 0.3, 0.5, 0.7, 0.9]
N_samples, fs_rate = 30000, 1000
segment_length = int(fs_rate)
num_segments = int(N_samples / segment_length)
T = 1.0 / fs_rate

xf = fftfreq(segment_length, T)[:segment_length//2]
index_20Hz = np.argmin(np.abs(xf - 20))
segment_centers = np.linspace(0, N_samples*T, num_segments, endpoint=False) + 0.5

# Dicionário de Tamanhos de Amostra para o Monte Carlo
amostras_monte_carlo = [100, 1000]
tabelas_latex = {}

for n_sim in amostras_monte_carlo:
    print(f"Executando simulação de Monte Carlo com {n_sim} iterações...")

    # Inicialização da estrutura de dados
    resultados = {win: {nl: {"sem_leak": [], "com_leak": []} for nl in noise_levels} for win in windows}

    for seed in range(1, n_sim + 1):
        np.random.seed(seed)

        # Gerar os sinais base da iteração para evitar reprocessamento
        sinais_ruido = {nl: {
            "sem_leak": generate_signal_base(False, N_samples, fs_rate, nl),
            "com_leak": generate_signal_base(True, N_samples, fs_rate, nl)
        } for nl in noise_levels}

        for win, cf in zip(windows, correction_calc):
            w = get_window(win, segment_length)

            for nl in noise_levels:
                for lk_str, lk_bool in [("sem_leak", False), ("com_leak", True)]:
                    x_s, y_s, flare_s = sinais_ruido[nl][lk_str]

                    peaks = []
                    for seg in range(num_segments):
                        start = seg * segment_length
                        end = start + segment_length
                        y_segment = y_s[start:end] * w
                        yf = fft(y_segment)
                        yf_mag = 2.0/segment_length * np.abs(yf[:segment_length//2]) * cf
                        peaks.append(yf_mag[index_20Hz])

                    flare_interp = np.interp(segment_centers, x_s, flare_s)
                    erro = np.sqrt(np.mean((np.array(peaks) - flare_interp)**2))*100
                    resultados[win][nl][lk_str].append(erro)

    # --- GERAÇÃO DOS PLOTS DO RMSE ---
    plt.figure(figsize=(12, 8))
    plt.rcParams.update({'font.size': 18})

    for win, name in zip(windows, window_names):
        means_lk = [np.mean(resultados[win][nl]["com_leak"]) for nl in noise_levels]
        stds_lk = [np.std(resultados[win][nl]["com_leak"]) for nl in noise_levels]

        plt.errorbar(noise_levels, means_lk, yerr=stds_lk, fmt='o-', capsize=5, label=name, lw=2)

    plt.title(f"Comparação RMSE por nível de ruído - sinal flare - ({n_sim} x)")
    #plt.title(f"RMSE comparison by Window type and noise level - simulated flare signal - {n_sim}x", fontsize=22, pad=15)
    plt.xlabel("Nível de ruído (branco + rosa)")
    plt.ylabel("RMSE (%)")
    plt.xticks(noise_levels)
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.legend()
    plt.tight_layout()
    plt.savefig(f"rmse_flare_{n_sim}_amostras.png", dpi=300)
    plt.show()

    # --- GERAR TEXTO LATEX ---
    latex_str = f"\\begin{{longtable}}{{cccc}}\n"
    latex_str += f"\\caption{{RMSE do Sinal Flare — {n_sim} amostras de Monte Carlo}} \\\\\n"
    latex_str += f"\\label{{tab:rmse_flare_{n_sim}}}\\\\\n"
    latex_str += f"\\toprule\nJanela & Ruído & Corrigido s/Leakage & Corrigido c/Leakage \\\\\n\\midrule\n\\endfirsthead\n"
    latex_str += f"\\caption[]{{RMSE do Sinal Flare — {n_sim} amostras (continuação)}} \\\\\n\\toprule\nJanela & Ruído & Corrigido s/Leakage & Corrigido c/Leakage \\\\\n\\midrule\n\\endhead\n\\bottomrule\n\\endfoot\n"

    for win, name in zip(windows, window_names):
        latex_str += f"% Janela {name}\n"
        latex_str += f"\\multirow{{5}}{{*}}{{{name}}}\n"
        for nl in noise_levels:
            m_nl, s_nl = np.mean(resultados[win][nl]["sem_leak"]), np.std(resultados[win][nl]["sem_leak"])
            m_lk, s_lk = np.mean(resultados[win][nl]["com_leak"]), np.std(resultados[win][nl]["com_leak"])
            latex_str += f" & {nl:.1f} & {m_nl:.5f} $\\pm$ {s_nl:.5f} & {m_lk:.5f} $\\pm$ {s_lk:.5f} \\\\\n"
        latex_str += "\\midrule\n"
    latex_str += "\\end{longtable}\n"
    tabelas_latex[n_sim] = latex_str

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.fftpack import fft
from scipy.signal import get_window

# ==========================================
# CENTRAL DE CONTROLE: ALTERE AQUI SEU SINAL
# ==========================================
N_MONTE_CARLO = 1000  # Escolha aqui: 100 ou 1000 amostras

# Parâmetros estruturais
windows = ['rectangular', 'hamming', 'hann', 'bartlett', 'blackman', 'flattop']
window_names = ['Rectangular', 'Hamming', 'Hann', 'Bartlett', 'Blackman', 'Flat-top']
correction_calc = [1.0, 1.85, 2.0, 2.02, 2.40, 4.68]
noise_levels = [0.1, 0.3, 0.5, 0.7, 0.9]

N_samples = 30000
fs_rate = 1000
segment_length = int(fs_rate)
num_segments = int(N_samples / segment_length)
segment_centers = np.linspace(0, N_samples/fs_rate, num_segments, endpoint=False) + 0.5

# Configuração dos subplots para os 5 níveis de ruído
fig, axes = plt.subplots(5, 1, figsize=(14, 28), sharex=True)
plt.rcParams.update({'font.size': 16})

print(f"Iniciando reconstrução temporal baseada em {N_MONTE_CARLO} amostras de Monte Carlo...")

for idx, nl in enumerate(noise_levels):
    ax = axes[idx]

    # Dicionário para acumular as curvas de reconstrução de cada simulação
    acumulador_janelas = {win: [] for win in windows}
    flare_ideal = None

    # Loop de Monte Carlo para a reconstrução
    for seed in range(1, N_MONTE_CARLO + 1):
        np.random.seed(seed)

        # Gerar sinal ruidoso desta rodada específica
        x_s, y_s, flare_s = generate_signal_base(LEAK=False, N=N_samples, fs=fs_rate, noise_amp=nl)
        if flare_ideal is None:
            flare_ideal = np.interp(segment_centers, x_s, flare_s)

        for win, cf in zip(windows, correction_calc):
            w = get_window(win, segment_length)
            peaks_rodada = []

            for seg in range(num_segments):
                start = seg * segment_length
                end = start + segment_length
                y_segment = y_s[start:end] * w
                yf = fft(y_segment)
                yf_mag = 2.0 / segment_length * np.abs(yf[:segment_length//2]) * cf
                peaks_rodada.append(yf_mag[20])

            acumulador_janelas[win].append(peaks_rodada)

    # Plotar as curvas médias calculadas no Monte Carlo
    for win, name in zip(windows, window_names):
        # Transforma em array para facilitar a manipulação matemática
        matriz_sinais = np.array(acumulador_janelas[win])

        # Em vez de: media_temporal = np.mean(matriz_sinais, axis=0)
        # Use uma rodada única para ver o ruído agir:
        rodada_unica = matriz_sinais[0, :]  # Pega a primeira das 1000 simulações

        ax.plot(rodada_unica, 'o-', alpha=0.6, linewidth=1.8, markersize=4, label=name)

        # Opcional: Você pode plotar a média ou adicionar a barra de desvio padrão sombreada
        #ax.plot(media_temporal, 'o-', alpha=0.6, linewidth=1.8, markersize=4, label=name)

    # Plot do envelope original ideal (Sinal limpo de referência)
    ax.plot(flare_ideal, color="black", linewidth=2.0, label="Sinal original")

    # Títulos e labels estruturados
    #ax.set_title(f"Reconstrução Temporal Média — {N_MONTE_CARLO}x Monte Carlo (Ruído = {nl})", fontsize=13, fontweight='bold')
    ax.set_title(f"Sinal Flare (ruído = {nl})", fontsize=20)
    ax.set_xlabel("Tempo (s)", fontsize=18)
    ax.set_ylabel("Amplitude(V)",fontsize=18)
    ax.grid(True, linestyle='--', alpha=0.7)
    ax.legend(fontsize=18, loc='upper right')

#axes[-1].set_xlabel("Segmentos de Tempo (Amostras de Janelamento)")
plt.tight_layout()
plt.savefig(f"reconstrucao_flare_{N_MONTE_CARLO}_amostras.png", dpi=300, bbox_inches='tight')
plt.show()
print("Processo concluído e gráficos salvos com sucesso!")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.fftpack import fft, fftfreq
from scipy.signal import get_window
import colorednoise as cn

# ==============================================================================
# 1. FUNÇÕES BASE (ENVELOPE E GERADOR DE SINAL)
# ==============================================================================

def flare_curve(t, t_peak, rise_time, decay_time):
    """Gera o envelope assimétrico do sinal de flare."""
    rise = np.exp(-(t_peak - t)**2 / (2 * rise_time**2))
    decay = np.exp(-(t - t_peak) / decay_time)
    return np.where(t < t_peak, rise, decay)

def generate_signal_base(LEAK, N, fs, noise_amp):
    """Gera o sinal de chopper modulado pelo flare com ruídos branco e rosa."""
    T = 1.0 / fs
    df = 1 / (N * T)
    chopper_freq = 20.0
    rede_freq = 60.0

    FN1 = np.round(chopper_freq * N * T)
    if LEAK:
        FN1 += 0.5

    chopper = FN1 * df
    x = np.linspace(0.0, N * T, N, endpoint=False)

    t_peak, rise_time, decay_time = 10, 5, 10
    flare = flare_curve(x, t_peak, rise_time, decay_time)

    y0 = np.sin(chopper * 2.0 * np.pi * x) * flare
    phase = np.random.rand(1)
    y1 = 0.2 * np.sin(rede_freq * 2.0 * np.pi * x - phase)

    noise_white = np.random.normal(0.0, 1.0, N)
    pink_noise = cn.powerlaw_psd_gaussian(1, N)
    delta_ampl = 0.5 * np.cos(2.0 * np.pi * 2.0 * x)

    y = (y0 + y1 + noise_white * noise_amp + pink_noise * noise_amp) + delta_ampl
    return x, y, flare

# ==============================================================================
# 2. PARÂMETROS GLOBAIS DE CONFIGURAÇÃO
# ==============================================================================

windows = ['rectangular', 'hamming', 'hann', 'bartlett', 'blackman', 'flattop']
window_names = ['Rectangular', 'Hamming', 'Hann', 'Bartlett', 'Blackman', 'Flat-top']
correction_calc = [1.0, 1.85, 2.0, 2.02, 2.40, 4.68]
noise_levels = [0.1, 0.3, 0.5, 0.7, 0.9]

N_samples, fs_rate = 30000, 1000
segment_length = int(fs_rate)
num_segments = int(N_samples / segment_length)
T = 1.0 / fs_rate

xf = fftfreq(segment_length, T)[:segment_length//2]
index_20Hz = np.argmin(np.abs(xf - 20))
segment_centers = np.linspace(0, N_samples*T, num_segments, endpoint=False) + 0.5

# Defina aqui quais tamanhos de Monte Carlo deseja processar para as tabelas/erros
amostras_monte_carlo = [100, 1000]

# ==============================================================================
# 3. LOOP PRINCIPAL DE MONTE CARLO (CÁLCULO DE RMSE E RECONSTRUÇÃO)
# ==============================================================================

for n_sim in amostras_monte_carlo:
    print(f"\n>>> Processando Simulação de Monte Carlo com {n_sim} iterações...")

    # Estrutura para armazenar o RMSE final de cada rodada
    resultados = {win: {nl: {"sem_leak": [], "com_leak": []} for nl in noise_levels} for win in windows}

    # Estrutura para armazenar o histórico temporal de picos (usado na reconstrução)
    # Rastreia apenas o cenário sem_leak para simplificar os subplots dinâmicos
    historico_reconstrucao = {nl: {win: [] for win in windows} for nl in noise_levels}
    flare_ideal_ref = None

    for seed in range(1, n_sim + 1):
        np.random.seed(seed)

        # Pré-geração dos sinais da rodada para otimização de performance
        sinais_ruido = {nl: {
            "sem_leak": generate_signal_base(False, N_samples, fs_rate, nl),
            "com_leak": generate_signal_base(True, N_samples, fs_rate, nl)
        } for nl in noise_levels}

        for win, cf in zip(windows, correction_calc):
            w = get_window(win, segment_length)

            for nl in noise_levels:
                for lk_str in ["sem_leak", "com_leak"]:
                    x_s, y_s, flare_s = sinais_ruido[nl][lk_str]

                    if flare_ideal_ref is None and lk_str == "sem_leak":
                        flare_ideal_ref = np.interp(segment_centers, x_s, flare_s)

                    peaks = []
                    for seg in range(num_segments):
                        start = seg * segment_length
                        end = start + segment_length
                        y_segment = y_s[start:end] * w
                        yf = fft(y_segment)
                        yf_mag = 2.0/segment_length * np.abs(yf[:segment_length//2]) * cf
                        peaks.append(yf_mag[index_20Hz])

                    flare_interp = np.interp(segment_centers, x_s, flare_s)
                    erro = np.sqrt(np.mean((np.array(peaks) - flare_interp)**2))*100
                    resultados[win][nl][lk_str].append(erro)

                    # Guarda o vetor de picos temporal para a plotagem posterior
                    if lk_str == "sem_leak":
                        historico_reconstrucao[nl][win].append(peaks)

    # --- GERAR GRÁFICO 1: EVOLUÇÃO DO RMSE ---
    plt.figure(figsize=(12, 8))
    plt.rcParams.update({'font.size': 16})
    for win, name in zip(windows, window_names):
        means_lk = [np.mean(resultados[win][nl]["com_leak"]) for nl in noise_levels]
        stds_lk = [np.std(resultados[win][nl]["com_leak"]) for nl in noise_levels]
        plt.errorbar(noise_levels, means_lk, yerr=stds_lk, fmt='o-', capsize=4, label=name, lw=1.8)

    #plt.title(f"Evolução do RMSE vs Nível de Ruído ({n_sim} Amostras - Com Leakage)")
    plt.title(f"RMSE comparison by Window type and noise - simulated flare signal ({n_sim} samples)")
    #plt.xlabel("Nível de Ruído (Branco + Rosa)")
    plt.xlabel('Noise level (white and pink)', fontsize=14)
    #plt.ylabel("RMSE Médio na Amplitude (V)")
    plt.ylabel('RMSE (%)', fontsize=16)
    plt.xticks(noise_levels)
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.legend()
    plt.tight_layout()
    plt.savefig(f"rmse_evolucao_flare_{n_sim}_amostras.png", dpi=300)
    plt.show()

    # --- GERAR TEXTO LATEX COMPATIVEL COM LONGTABLE ---
    latex_str = f"\\begin{{longtable}}{{cccc}}\n"
    latex_str += f"\\caption{{RMSE do Sinal Flare — {n_sim} amostras de Monte Carlo}} \\\\\n"
    latex_str += f"\\label{{tab:rmse_flare_{n_sim}}}\\\\\n"
    latex_str += f"\\toprule\nJanela & Ruído & Corrigido s/Leakage & Corrigido c/Leakage \\\\\n\\midrule\n\\endfirsthead\n"
    latex_str += f"\\caption[]{{RMSE do Sinal Flare — {n_sim} amostras (continuação)}} \\\\\n\\toprule\nJanela & Ruído & Corrigido s/Leakage & Corrigido c/Leakage \\\\\n\\midrule\n\\endhead\n\\bottomrule\n\\endfoot\n"

    for win, name in zip(windows, window_names):
        latex_str += f"% Janela {name}\n"
        latex_str += f"\\multirow{{5}}{{*}}{{{name}}}\n"
        for nl in noise_levels:
            m_nl, s_nl = np.mean(resultados[win][nl]["sem_leak"]), np.std(resultados[win][nl]["sem_leak"])
            m_lk, s_lk = np.mean(resultados[win][nl]["com_leak"]), np.std(resultados[win][nl]["com_leak"])
            latex_str += f" & {nl:.1f} & {m_nl:.5f} $\\pm$ {s_nl:.5f} & {m_lk:.5f} $\\pm$ {s_lk:.5f} \\\\\n"
        latex_str += "\\midrule\n"
    latex_str += "\\end{longtable}\n"

    print(f"\n--- TABELA LATEX GERADA PARA {n_sim} AMOSTRAS ---")
    print(latex_str)

    # --- GERAR GRÁFICO 2: RECONSTRUÇÃO TEMPORAL INSTANTÂNEA (OPÇÃO A CONCISA) ---
    fig, axes = plt.subplots(5, 1, figsize=(13, 22), sharex=True)

    for idx, nl in enumerate(noise_levels):
        ax = axes[idx]

        for win, name in zip(windows, window_names):
            matriz_sinais = np.array(historico_reconstrucao[nl][win])

            # OPÇÃO A: Extrai estritamente a primeira rodada (Single Shot) para expor a distorção do ruído
            rodada_instantanea = matriz_sinais[0, :]

            ax.plot(rodada_instantanea, 'o-', alpha=0.6, linewidth=1.5, markersize=4, label=name)

        ax.plot(flare_ideal_ref, color="black", linewidth=2.5, label="Original signal")
        #ax.set_title(f"Reconstrução Temporal Instantânea (Single Shot) — Ruído = {nl} (MC: {n_sim}x)", fontsize=12, fontweight='bold')
        ax.set_title(f"Sinal flare (nível de ruído: {nl})", fontsize=20)
        ax.set_ylabel("Amplitude (V)")
        ax.set_xlabel("Amostras")
        ax.grid(True, linestyle='--', alpha=0.5)

        if idx == 0:
            ax.legend(loc="upper right", bbox_to_anchor=(1.16, 1.0))


    plt.tight_layout()
    plt.savefig(f"reconstrucao_instantanea_flare_{n_sim}_amostras.png", dpi=300, bbox_inches='tight')
    plt.show()

print("\n>>> Execução de todo o pipeline concluída com sucesso!")